## Pendahuluan

Regresi linear adalah fondasi dari analisis ekonometrika. Tutorial ini membahas cara membangun model regresi linear di R secara lengkap — mulai dari memuat data, estimasi OLS, interpretasi output, hingga uji asumsi klasik (normalitas, heteroskedastisitas, multikolinearitas, dan autokorelasi).

### Paket yang Digunakan

```r
library(tidyverse)  # manipulasi & visualisasi data
library(broom)      # tidying model output
library(lmtest)     # uji diagnostik (Breusch-Pagan, Durbin-Watson)
library(car)        # VIF untuk multikolinearitas
library(performance)# check_model() untuk diagnostik visual
```

## Data

Kita gunakan dataset bawaan `mtcars` untuk memprediksi konsumsi bahan bakar (`mpg`) berdasarkan beberapa fitur kendaraan.

```r
data(mtcars)

df <- mtcars |>
  select(mpg, hp, wt, qsec, am) |>
  mutate(am = factor(am, labels = c("Otomatis", "Manual")))

glimpse(df)
```

## Estimasi Model OLS

Model regresi linear berganda:

$$
\text{mpg}_i = \beta_0 + \beta_1 \text{hp}_i + \beta_2 \text{wt}_i + \beta_3 \text{qsec}_i + \beta_4 \text{am}_i + \varepsilon_i
$$

```r
model <- lm(mpg ~ hp + wt + qsec + am, data = df)
summary(model)
```

### Output yang Rapi dengan `broom`

```r
# Koefisien
tidy(model, conf.int = TRUE)

# Goodness of fit
glance(model)

# Nilai fitted + residual per observasi
augment(model)
```

## Interpretasi Koefisien

| Variabel | Interpretasi |
|----------|-------------|
| `hp` | Setiap kenaikan 1 HP, `mpg` turun sebesar $\hat{\beta}_1$, *ceteris paribus* |
| `wt` | Setiap kenaikan 1000 lbs berat, `mpg` turun sebesar $\hat{\beta}_2$ |
| `am` | Mobil manual memiliki `mpg` lebih tinggi/rendah sebesar $\hat{\beta}_4$ dibanding otomatis |

> **Tips:** Gunakan `tidy(model, conf.int = TRUE)` untuk melihat confidence interval setiap koefisien.

## Uji Asumsi Klasik

### 1. Normalitas Residual

```r
# Shapiro-Wilk test
shapiro.test(residuals(model))

# Visual: Q-Q plot
ggplot(augment(model), aes(sample = .resid)) +
  stat_qq() +
  stat_qq_line(color = "steelblue") +
  labs(title = "Q-Q Plot Residual", x = "Theoretical", y = "Sample")
```

### 2. Heteroskedastisitas

```r
# Breusch-Pagan test
bptest(model)

# Visual: residual vs fitted
ggplot(augment(model), aes(.fitted, .resid)) +
  geom_point(alpha = 0.6) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  geom_smooth(se = FALSE, color = "steelblue") +
  labs(title = "Residual vs Fitted", x = "Fitted Values", y = "Residuals")
```

### 3. Multikolinearitas

```r
# Variance Inflation Factor
vif(model)
```

> **Rule of thumb:** VIF > 10 mengindikasikan multikolinearitas serius.

### 4. Autokorelasi

```r
# Durbin-Watson test
dwtest(model)
```

## Visualisasi Hasil

### Coefficient Plot

```r
tidy(model, conf.int = TRUE) |>
  filter(term != "(Intercept)") |>
  ggplot(aes(x = estimate, y = reorder(term, estimate))) +
  geom_point(size = 3) +
  geom_errorbarh(aes(xmin = conf.low, xmax = conf.high), height = 0.2) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Coefficient Plot",
    x = "Estimasi Koefisien (95% CI)",
    y = NULL
  )
```

### Diagnostik Visual Lengkap

```r
# Satu fungsi untuk semua plot diagnostik
performance::check_model(model)
```

## Ringkasan

| Langkah | Fungsi/Paket | Tujuan |
|---------|-------------|--------|
| Estimasi | `lm()` | Fit model OLS |
| Output rapi | `broom::tidy()`, `glance()` | Koefisien & GOF |
| Normalitas | `shapiro.test()` | Uji distribusi residual |
| Heteroskedastisitas | `lmtest::bptest()` | Uji varians konstan |
| Multikolinearitas | `car::vif()` | Cek korelasi antar prediktor |
| Autokorelasi | `lmtest::dwtest()` | Uji korelasi serial |
| Diagnostik visual | `performance::check_model()` | Semua plot sekaligus |

---

*Tutorial selanjutnya: Regresi Panel Data dengan `plm` di R.*